In [ ]:
import pandas as pd
import requests
import time
import os
import zipfile
import urllib.request
from tqdm import tqdm 
from dotenv import load_dotenv, find_dotenv


In [ ]:

# Tự động tìm file .env ở các thư mục cha
load_dotenv(find_dotenv())

# Sử dụng biến bình thường
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
print(TMDB_API_KEY)

In [ ]:
DATA_PATH = "../data"

In [ ]:
# !pip install tqdm

In [ ]:
# !pip3 install kagglehub 

In [ ]:
# !pip install kagglehub[hf-datasets]

In [ ]:
import kagglehub




MOVIELENS_DIR = os.path.join(DATA_PATH, 'movielens')
path = ""
if not os.path.exists(MOVIELENS_DIR):
    # Download latest version
    path = kagglehub.dataset_download("grouplens/movielens-latest-full")
else:
    path = MOVIELENS_DIR
    
print("Path to dataset files:", path)

In [ ]:
full_df = {
    "ratings": pd.read_csv(path + "/ratings.csv"),
    "tags": pd.read_csv(path + "/tags.csv"),
    "links": pd.read_csv(path + "/links.csv"),
    "movies": pd.read_csv(path + "/movies.csv"),
    "genome_scores": pd.read_csv(path + "/genome_scores.csv"),
    "genome_tags": pd.read_csv(path + "/genome_tags.csv")
}

In [ ]:
print('Số phim: ', full_df['movies'].shape[0]) 
print('Số người dùng: ', full_df['ratings']['userId'].nunique())
print('Số lượt đánh giá: ', full_df['ratings'].shape[0])

In [ ]:
for name, df in full_df.items():
    print(f"{50*'='}")
    print(f"DataFrame '{name}':")
    print(df.head())
    
    save_path = os.path.join(MOVIELENS_DIR, f"{name}.csv")
    df.to_csv(save_path, index=False)
    print("\n")

In [28]:
# 1. Nhóm theo movieId để tính Số lượt đánh giá và Điểm trung bình
movie_stats = full_df['ratings'].groupby('movieId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean')
).reset_index()

# 2. Áp dụng tiêu chuẩn lọc (Filter)
MIN_RATINGS = 10  # Phim phải có ít nhất 50 người rate
MIN_SCORE = 2.0   # Điểm trung bình từ 3.0 trở lên

quality_movies = movie_stats[
    (movie_stats['rating_count'] >= MIN_RATINGS) & 
    (movie_stats['rating_mean'] >= MIN_SCORE)
]

print(f"Số phim ban đầu: {len(movie_stats)}")
print(f"Số phim 'Đáng xem' sau khi lọc: {len(quality_movies)}")

# 3. Gộp (Merge) kết quả đã lọc với bảng links_df để lấy tmdbId
filtered_links_df = pd.merge(quality_movies, full_df['links'], on='movieId', how='inner')

display(filtered_links_df.sort_values(by='rating_count', ascending=False).head())

Số phim ban đầu: 53889
Số phim 'Đáng xem' sau khi lọc: 23226


,movieId,rating_count,rating_mean,imdbId,tmdbId
314,318,97999,4.424188,111161,278.0
351,356,97040,4.056585,109830,13.0
292,296,92406,4.173971,110912,680.0
585,593,87899,4.151412,102926,274.0
2439,2571,84545,4.149695,133093,603.0


In [ ]:
MIN_RELEASE_YEAR = 1995

def fetch_movie_vietnamese_metadata(tmdb_id):
    """Hàm gọi API TMDB để lấy thông tin phim bằng tiếng Việt"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=vi-VN"
    
    try:
        response = requests.get(url, timeout=10)
        # Nếu request thành công
        if response.status_code == 200:
            data = response.json()
            
            # check date sau 1995
            if 'release_date' in data and data['release_date']:
                release_year = int(data['release_date'].split('-')[0])
                if release_year < MIN_RELEASE_YEAR:
                    return None
            is_fetch_again = False
            # Nếu không có tiếng việt, gọi lại với ngôn ngữ mặc định (tiếng Anh)
            if not data.get('overview') or data.get('overview') == '':
                url_en = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=en-US"
                response_en = requests.get(url_en, timeout=10)
                if response_en.status_code == 200:
                    data_en = response_en.json()
                    # data['title'] = data_en.get('title', '')
                    data['overview'] = data_en.get('overview', '')
                    is_fetch_again = True
            
            return {
                'tmdbId': tmdb_id,
                'title_vi': data.get('title', ''),
                'overview_vi' :is_fetch_again == False and data.get('overview', '') or '', # Nếu trống thì sẽ dịch ở giai đoạn sau
                # Lấy danh sách tên thể loại và nối thành chuỗi
                'genres': ", ".join([g['name'] for g in data.get('genres', [])]) if data.get('genres') else "",
                'release_date': data.get('release_date', ''),
                'poster_path': data.get('poster_path', '') # Rất hữu ích cho frontend sau này,
                
            } | data
        # Nếu phim không tồn tại trên TMDB
        elif response.status_code == 404:
            return None
    except Exception as e:
        print(f"Lỗi khi fetch tmdbId {tmdb_id}: {e}")
        return None
    return None

In [ ]:
tmdb_id = 91550 
metadata = fetch_movie_vietnamese_metadata(tmdb_id)
for k, v in metadata.items():
    print(f"{k}: {v}")


In [33]:
movie_vi_data = []

# Nơi lưu phim
TMDB_DIR = os.path.join(DATA_PATH, 'tmdb')
if not os.path.exists(TMDB_DIR):
    os.makedirs(TMDB_DIR)
    
# Kiểm tra đã lưu trước hay chưa
if os.path.exists(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl")):
    print("Dữ liệu phim tiếng Việt đã tồn tại, không cần fetch lại.")
    movie_vi_data = pd.read_json(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl"), lines=True).to_dict(orient='records')

# fetch theo đoạn: limit, offset
last_offset = 0
batch_size = 1000


def fetch_movie_batch(limit=1000, offset=0):
    """Hàm fetch một batch phim từ filtered_links_df"""
    batch_links = filtered_links_df.iloc[offset:offset+limit]
    batch_data = [] 
    
    for index, row in tqdm(batch_links.iterrows(), total=batch_links.shape[0], desc=f"Đang fetch batch {offset + 1}-{offset + limit}"):
        movie_id = row['movieId']
        tmdb_id = row['tmdbId']
        
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)
        
        if metadata:
            metadata['movieId'] = movie_id
            batch_data.append(metadata)
        
        time.sleep(0.05)  # Tránh bị block API
    
    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + limit

def save_movie(data, offset, limit, prefix="movies_vietnamese_metadata", is_all=False):
    # Chuyển list các dictionary thành Pandas DataFrame
    movies_vi_df = pd.DataFrame(data)

    # Hiển thị vài dòng đầu để kiểm tra
    # display(movies_vi_df.head())

        
    start_index = offset + 1
    end_index = offset + len(data)
    if is_all:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}.jsonl")
    else:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}_{start_index}-{end_index}.jsonl")
    movies_vi_df.to_json(output_filename, orient='records', lines=True, force_ascii=False)
    print(f"\nĐã lưu thành công {len(movies_vi_df)} phim tiếng Việt vào file {output_filename}!")




In [34]:
for i in range(10):
    data, last_offset = fetch_movie_batch(limit=batch_size, offset=last_offset)

Đang fetch batch 1-1000:  62%|██████▏   | 618/1000 [13:38<27:29,  4.32s/it]

Lỗi khi fetch tmdbId 28121.0: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/28121.0?api_key=de6ad9963ce22f788c1d2180429def70&language=vi-VN (Caused by NameResolutionError("HTTPSConnection(host='api.themoviedb.org', port=443): Failed to resolve 'api.themoviedb.org' ([Errno 11001] getaddrinfo failed)"))


Đang fetch batch 1-1000: 100%|██████████| 1000/1000 [25:26<00:00,  1.53s/it]



Đã lưu thành công 489 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_1-489.jsonl!


Đang fetch batch 1001-2000: 100%|██████████| 1000/1000 [47:15<00:00,  2.84s/it] 


...


Đã lưu thành công 303 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_8001-8303.jsonl!


Đang fetch batch 9001-10000:  96%|█████████▌| 959/1000 [30:35<00:44,  1.09s/it]

Lỗi khi fetch tmdbId 9392.0: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Đang fetch batch 9001-10000: 100%|██████████| 1000/1000 [31:41<00:00,  1.90s/it]


Đã lưu thành công 582 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_9001-9582.jsonl!


In [36]:
save_movie(movie_vi_data, offset=0, limit=len(movie_vi_data), prefix="movies_vietnamese_metadata_0-10000", is_all=True)


Đã lưu thành công 4191 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_0-10000.jsonl!


In [37]:
movie_vi_data_df = pd.DataFrame(movie_vi_data)

In [48]:
movie_vi_data_df.count()

tmdbId                   4191
title_vi                 4191
overview_vi              4191
genres                   4191
release_date             4191
poster_path              4181
adult                    4191
backdrop_path            4016
belongs_to_collection     601
budget                   4191
homepage                 4191
id                       4191
imdb_id                  4191
origin_country           4191
original_language        4191
original_title           4191
overview                 4191
popularity               4191
production_companies     4191
production_countries     4191
revenue                  4191
runtime                  4191
softcore                 4191
spoken_languages         4191
status                   4191
tagline                  4191
title                    4191
video                    4191
vote_average             4191
vote_count               4191
movieId                  4191
dtype: int64

In [50]:
movie_vi_data_df.to_csv(os.path.join(TMDB_DIR, "movies_metadata_0-10000_1995-01-01.csv"), index=False, encoding='utf-8-sig')

In [38]:
movie_vi_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4191 entries, 0 to 4190
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tmdbId                 4191 non-null   float64
 1   title_vi               4191 non-null   str    
 2   overview_vi            4191 non-null   str    
 3   genres                 4191 non-null   object 
 4   release_date           4191 non-null   str    
 5   poster_path            4181 non-null   str    
 6   adult                  4191 non-null   bool   
 7   backdrop_path          4016 non-null   str    
 8   belongs_to_collection  601 non-null    object 
 9   budget                 4191 non-null   int64  
 10  homepage               4191 non-null   str    
 11  id                     4191 non-null   int64  
 12  imdb_id                4191 non-null   str    
 13  origin_country         4191 non-null   object 
 14  original_language      4191 non-null   str    
 15  original_title 

In [39]:
movie_vi_data_df = movie_vi_data_df.dropna(subset=['overview_vi'])

# 2. Tính trung bình số từ (length) của cột overview_vi
mean_length = movie_vi_data_df['overview_vi'].str.split().str.len().mean()
print("Trung bình length của overview_vi:", mean_length)


Trung bình length của overview_vi: 18.55810069195896


In [46]:
# 3. Lọc các phim có overview ngắn hơn 50 từ
short_movies_df = movie_vi_data_df[movie_vi_data_df['overview_vi'].str.split().str.len() == 0]
display(short_movies_df.sort_values('release_date', ascending=False).head(10))



# Lọc phim năm 1995 trở đi
recent_movies_df = short_movies_df[short_movies_df['release_date'] >= '2000-01-01']
display(recent_movies_df.sort_values('release_date', ascending=True).head(10))

,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
1955,110428.0,Camille Claudel 1915,,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2013-03-13,/jHU5CiEk4QoCrvxJV8uOLxi3y5y.jpg,False,/8uhoQGtmfDQzjPH7XsUcDZ0TT8e.jpg,None,3512454,...,95,False,"[{'english_name': 'French', 'iso_639_1': 'fr',...",Released,,Camille Claudel 1915,False,5.894,85,4484.0
526,141210.0,The Sleepover,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 27, 'n...",2012-10-12,/tdaZD1Db9CXu96oLdAHElJsBPfb.jpg,False,NaN,None,0,...,6,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Sleepover,False,6.500,11,1115.0
1393,171982.0,Romance,,"[{'id': 10749, 'name': 'Phim Lãng Mạn'}, {'id'...",2012-10-09,/hrXTHdoXr4SiP9Dvx1muQ1YCdus.jpg,False,NaN,None,0,...,27,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Romance,False,6.000,1,2894.0
1065,143750.0,The Farmer's Wife,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2012-06-20,NaN,False,NaN,None,0,...,18,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Farmer's Wife,False,10.000,1,2223.0
1861,136558.0,Kingdom Come,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 99, 'n...",2012-04-28,/70a4d1m1EiGpltPmtiCWMt3l0zK.jpg,False,NaN,None,0,...,88,False,[],Released,,Kingdom Come,False,6.200,5,4249.0
1284,84198.0,A Place at the Table,,"[{'id': 99, 'name': 'Phim Tài Liệu'}]",2012-03-22,/ptli8HtDrapZdVdBnYSN8zxJtw0.jpg,False,/hdFefzO4RPq4txKEbFU7EulePsS.jpg,None,0,...,84,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,A Place at the Table,False,6.593,27,2679.0
2758,66111.0,Luster,,"[{'id': 53, 'name': 'Phim Gây Cấn'}, {'id': 27...",2010-12-12,/nqwnFnnxRahBDzDDiPDgD1GTnne.jpg,False,/5QY8hsHSW3FojbVKikOqqJ8pZnZ.jpg,None,0,...,81,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Luster,False,4.500,14,6712.0
464,79782.0,Wenecja,,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2010-06-11,/jk8PwaZ14Duxy8hmLbYLFcYTUP9.jpg,False,/8vrmhLhPucmEOJr6EDxhxzVDMXi.jpg,None,1783810,...,110,False,"[{'english_name': 'Czech', 'iso_639_1': 'cs', ...",Released,,Wenecja,False,7.139,18,895.0
2495,99885.0,Divine Intervention,,"[{'id': 27, 'name': 'Phim Kinh Dị'}, {'id': 53...",2007-02-26,/aGNSL9b1GZBYWKfITVQQkmKoq76.jpg,False,NaN,None,0,...,87,False,[],Released,,Divine Intervention,False,0.000,0,6017.0
3671,3509.0,A Scanner Darkly,,"[{'id': 16, 'name': 'Phim Hoạt Hình'}, {'id': ...",2006-07-07,/lUKudOpHICDj6A6SO7DdaZM4W48.jpg,False,/do8idarfaUTxxwaarefTNA724Qj.jpg,None,8700000,...,100,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,A Scanner Darkly,False,6.812,2023,27904.0


,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
1577,19214.0,Passion of Mind,,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2000-01-07,/A7lzVd2YHz85pgOugGJCJ983bQB.jpg,False,/vyqmGMafFwTa8phpx234MGFIuE6.jpg,None,12000000,...,105,False,"[{'english_name': 'French', 'iso_639_1': 'fr',...",Released,,Passion of Mind,False,5.816,49,3540.0
2513,12490.0,Ordinary Decent Criminal,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 53, 'n...",2000-01-07,/jZiORaxALVDGoMUBau85E4CWB0d.jpg,False,/sHnvSBVKjbkqTXFRgDbaBfGKIOL.jpg,None,0,...,93,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Ordinary Decent Criminal,False,6.165,254,6067.0
1485,10471.0,Next Friday,,"[{'id': 35, 'name': 'Phim Hài'}]",2000-01-12,/dP4fYCGQZzg17ta7FvLwJVCwwt9.jpg,False,/xi4Uy11TdACds3SXglrbbmaVd90.jpg,"{'id': 43563, 'name': 'Friday Collection', 'po...",11000000,...,98,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Next Friday,False,6.530,676,3177.0
1497,17908.0,My Dog Skip,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",2000-01-12,/nnPv7RuSn6UaPjZussIFljZdWS8.jpg,False,/mrhdbBV44O87ikOObJpIMuHoMNL.jpg,None,6000000,...,95,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,My Dog Skip,False,6.710,293,3189.0
1498,10384.0,Supernova,,"[{'id': 878, 'name': 'Phim Khoa Học Viễn Tưởng...",2000-01-14,/AmHGUjhgXnOYRYrF2ZYMtzfwxPe.jpg,False,/przsU4aiGsdlDkCp5uV8kq8gbLe.jpg,None,90000000,...,91,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Supernova,False,4.989,451,3190.0
3356,10213.0,From Dusk Till Dawn 3: The Hangman's Daughter,,"[{'id': 37, 'name': 'Phim Miền Tây'}, {'id': 2...",2000-01-18,/io5lQ7VRJGBCg2kTJ6BRWRclqJE.jpg,False,/n4ChV0MskyVOzOMBO8V6xoi6Z94.jpg,"{'id': 10924, 'name': 'From Dusk Till Dawn Col...",5000000,...,94,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,From Dusk Till Dawn 3: The Hangman's Daughter,False,4.917,405,27004.0
2761,29507.0,Erleuchtung garantiert,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",2000-01-20,/aLH616UWPUot9t8KWzk8DeoQwCM.jpg,False,/j13gcHwSsW7j4ulY6q5aUOSRvvs.jpg,None,0,...,109,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Erleuchtung garantiert,False,6.900,23,6717.0
1500,10472.0,Down to You,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",2000-01-21,/pqEEtX4S8j2XqZ9JAD97fz2DPkV.jpg,False,/kqcbwRrUmkcUe6TcFWHMnobVDon.jpg,None,9000000,...,91,False,"[{'english_name': 'French', 'iso_639_1': 'fr',...",Released,,Down to You,False,5.200,271,3225.0
1502,97805.0,Wirey Spindell,,"[{'id': 35, 'name': 'Phim Hài'}]",2000-01-21,/4G8TwEngXbAnuLKQvOZ2qbODnh5.jpg,False,/1x3zmnVHEYvX7gd6xgp22NfhPRL.jpg,None,0,...,101,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Wirey Spindell,False,4.700,7,3228.0
1568,125709.0,Joe Gould's Secret,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2000-01-21,/n7X4jmmRIRtqNQWXJt8LxZcS1Ja.jpg,False,/1VYE83DZuMrkmc5w364oeYDzYwK.jpg,None,0,...,104,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Joe Gould's Secret,False,6.000,21,3514.0


In [47]:
recent_movies_df.count()
# short_movies_df.count()

tmdbId                   1815
title_vi                 1815
overview_vi              1815
genres                   1815
release_date             1815
poster_path              1811
adult                    1815
backdrop_path            1734
belongs_to_collection     179
budget                   1815
homepage                 1815
id                       1815
imdb_id                  1815
origin_country           1815
original_language        1815
original_title           1815
overview                 1815
popularity               1815
production_companies     1815
production_countries     1815
revenue                  1815
runtime                  1815
softcore                 1815
spoken_languages         1815
status                   1815
tagline                  1815
title                    1815
video                    1815
vote_average             1815
vote_count               1815
movieId                  1815
dtype: int64